<a href="https://colab.research.google.com/github/juliocnsouzadev/notebooks/blob/issue-17-Deep_Learning_for_Text_with_PyTorch/PyTorch/Deep_Learning_for_Text_With_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Text with PyTorch

In [ ]:
!pip uninstall torch -y
!pip install torch==2.0.0+cpu

!pip uninstall torchtext -y
!pip install torchtext==0.15.1

!pip uninstall nltk -y
!pip install nltk==3.6.5

!pip uninstall spacy -y
!pip install spacy==3.1.3

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

## Dataset and DataLoader

In [ ]:
# Import the necessary libraries
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = targets

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index], self.targets[index]
    

### Helper Functions

In [ ]:
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Assuming you have a tokenizer and stemmer defined
tokenizer = word_tokenize
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_sentences(sentences):
    processed_sentences = []
    for sentence in sentences:
        sentence = sentence.lower()
        tokens = tokenizer(sentence)
        tokens = [token for token in tokens if token not in stop_words]
        tokens = [stemmer.stem(token) for token in tokens]
        freq_dist = FreqDist(tokens)
        threshold = 2
        tokens = [token for token in tokens if freq_dist[token] > threshold]
        processed_sentences.append(' '.join(tokens))
    return processed_sentences

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def encode_sentences(sentences):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(sentences)
    encoded_sentences = X.toarray()
    return encoded_sentences, vectorizer

In [ ]:
import re

def extract_sentences(data):
    sentences = re.findall(r'[A-Z][^.!?]*[.!?]', data)
    return sentences

## Text Processing Pipeline

In [ ]:
from torch.utils.data import DataLoader


def text_processing_pipeline(text):
    tokens = preprocess_sentences(text)
    encoded_sentences, vectorizer = encode_sentences(tokens)
    dataset = TextDataset(encoded_sentences)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
    return dataloader, vectorizer

In [ ]:
text_data = "In the city of Dataville, a data analyst named Alex explores hidden insights within vast data. With determination, Alex uncovers patterns, cleanses the data, and unlocks innovation. Join this adventure to unleash the power of data-driven decisions."
sentences = extract_sentences(text_data)

dataloaders, vectorizer = [text_processing_pipeline(text) for text in sentences]
print(next(iter(dataloaders))[0,:10])